# Maximum cut

## Problem description

Consider an undirected graph $G = (V, E)$ (assume $|V| = n$) with weights on the edges in $E$. The goal of the Max-Cut problem is to find a cut (partition nodes into $S$ and $V\setminus S$) so that the sum of the weights of the cut edges is maximised.

## Problem formulation 1

The model variables are the following:
* $v_i \in \{0,1\}$: Binary variable indicating if vertex $i$ is in $S$.

The objective is:

$\max f(v) = \sum_{(i,j)\in E} w_{ij} |v_i - v_j|$,

where $w_{ij}$ is the weight of the edge connecting node $i$ with node $j$.

## Problem formulation 2

The model variables are the following:
* $v_i \in \{-1,1\}$: Integer variable indicating if vertex $i$ is in $S$ ($v_i = 1$) or not ($v_i = -1$).

The objective is:

$\max f(v) = \frac{1}{4}\sum_{i = 1}^n \sum_{j = 1}^n w_{ij} (1 - v_i\cdot v_j)$,

where $w_{ij}$ is the weight of the edge connecting node $i$ with node $j$.

We can relax the domain of the variables to be continuous and impose the following constraints:

$v_i^2 = 1, \quad \forall i \in \{1, \dots, n\}.$

## Required packages

In [1]:
import random
import gurobipy as gp
import networkx as nx
from itertools import combinations
import matplotlib.pyplot as plt
import scipy # Used within Gurobi

## Data generation

The following code generates random connected graph. The value for `number_of_nodes`, `edge_density` and `max_edge_weight` can be changed.

In [2]:
# The values for the graph
number_of_nodes = 10
edge_density = 0.5      # Should be in [0,1]
max_edge_weight = 10
random.seed(2026)

# Correcting edge_density if not in [0,1]
edge_density = max(min(edge_density, 1), 1/number_of_nodes)
# Creating edge weight options
edge_weights = [i for i in range(1,max(1,max_edge_weight))]

# Generating nodes of graph
G = nx.Graph()
for i in range(number_of_nodes):
    G.add_node(i)

# Creating set of all connections
edge_set = set(combinations(range(number_of_nodes), 2))

# Connecting all nodes (unconnected components are boring)
for i in range(number_of_nodes-1):
    G.add_edge(i, i+1, weight=random.choice(edge_weights))
    edge_set.remove((i,i+1))

# Randomly choose edges to add until density reached
while True:
    edge = random.choice(list(edge_set))
    G.add_edge(edge[0],edge[1], weight=random.choice(edge_weights))
    edge_set.remove(edge)
    if nx.density(G) > edge_density:
        break

The following code provides a preview of the graph.

In [ ]:
pos = nx.spring_layout(G)
nx.draw(G, pos)

## Model building and solving (Formulation 1)

The following code builds and solves the first model formulation using the Gurobi API for Python.

In [3]:
# Initialise model
model = gp.Model()

# Add variables
v = model.addVars(number_of_nodes, name="v", vtype="b")
e = {}
e_plus = {}
for i, edge in enumerate(G.edges):
    e[i] = model.addVar(name=f"e_{edge[0]}_{edge[1]}", lb = -float('inf'))
    e_plus[i] = model.addVar(name=f"e_{edge[0]}_{edge[1]}", lb = -float('inf'))
    model.addConstr(e[i] == v[edge[0]] - v[edge[1]])
    model.addConstr(e_plus[i] == gp.abs_(e[i]))

# Add objective function
model.setObjective(gp.quicksum(G.get_edge_data(edge[0],edge[1])["weight"]*e_plus[i] for i in e), gp.GRB.MAXIMIZE)

# Optimize model
model.optimize()

Set parameter Username
Set parameter LicenseID to value 2751469
Academic license - for non-commercial use only - expires 2026-12-08
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 23 rows, 56 columns and 69 nonzeros (Max)
Model fingerprint: 0x838f9ce4
Model has 23 linear objective coefficients
Model has 23 simple general constraints
  23 ABS
Variable types: 46 continuous, 10 integer (10 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [9e+00, 9e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [0e+00, 0e+00]

Found heuristic solution: objective -0.0000000
Presolve added 69 rows and 46 columns
Presolve time: 0.00s
Presolved: 92 rows, 102 columns, 276 nonzeros
Variable types: 69 continuous, 33 integer (33 binary)

R

## Solution handling

The following code collects and prints the solution.

In [ ]:
# Collecting the partition
S_set = set()
not_S_set = set()
for i in range(number_of_nodes):
    if v[i].X == 1:
        S_set.add(i)
    else:
        not_S_set.add(i)
print(f"S: {S_set}")
print(f"V minus S: {not_S_set}")

# Collecting the cut edges
cut_edges = set()
for edge in e_plus:
    variable = e_plus[edge]
    if variable.X == 1:
        v1 = int(variable.VarName.split("_")[1])
        v2 = int(variable.VarName.split("_")[2])
        cut_edges.add((v1,v2))
print(f"Cut edges: {cut_edges}")

The following code plots the graph from above colouring the nodes into the partition. Cut edges are highlighted.

In [ ]:
# Nodes
nx.draw_networkx_nodes(G, pos, nodelist=list(S_set), node_color="tab:red")
nx.draw_networkx_nodes(G, pos, nodelist=list(not_S_set), node_color="tab:blue")

# Edges
nx.draw_networkx_edges(G, pos, width=1.0, alpha=0.5)
nx.draw_networkx_edges(G, pos, edgelist=list(cut_edges), width=5, alpha=0.5, edge_color="tab:red")

# Plot figure
plt.tight_layout()
plt.axis("off")
plt.show()

## Model building and solving (Formulation 2)

The following code builds and solves the second model formulation using the Gurobi API for Python.

In [6]:
# Initialise model
model = gp.Model()

# Add variables
v = model.addVars(number_of_nodes, name="v", vtype="c", lb = -float('inf'))
e = {}

# Constraints
model.addConstrs(v[i]*v[i] == 1 for i in range(number_of_nodes))
for i, edge in enumerate(G.edges):
    e[i] = model.addVar(name=f"e_{edge[0]}_{edge[1]}", lb = -float('inf'))
    model.addConstr(e[i] == 1 - v[edge[0]]*v[edge[1]])

# Add objective function
# (Note a slight change in scaling due to not double counting)
model.setObjective(0.5*gp.quicksum(G.get_edge_data(edge[0],edge[1])["weight"]*e[i] for i in e), gp.GRB.MAXIMIZE)

# Optimize model
model.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 0 rows, 33 columns and 0 nonzeros (Max)
Model fingerprint: 0x19c324b2
Model has 23 linear objective coefficients
Model has 33 quadratic constraints
Coefficient statistics:
  Matrix range     [0e+00, 0e+00]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [5e+00, 5e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [0e+00, 0e+00]
  QRHS range       [1e+00, 1e+00]


Continuous model is non-convex -- solving as a MIP


Presolve removed 0 rows and 23 columns
Presolve time: 0.00s
Presolved: 17 rows, 27 columns, 51 nonzeros
Variable types: 0 continuous, 27 integer (27 binary)
Found heuristic solution: objective 153.0000000

Root relaxation: objective 1.800000e+02, 10 iter

## Solution handling

The following code collects and prints the solution.

In [ ]:
# Collecting the partition
S_set = set()
not_S_set = set()
for i in range(number_of_nodes):
    if v[i].X == 1:
        S_set.add(i)
    else:
        not_S_set.add(i)
print(f"S: {S_set}")
print(f"V minus S: {not_S_set}")

# Collecting the cut edges
cut_edges = set()
for edge in e:
    variable = e[edge]
    if variable.X != 0:
        v1 = int(variable.VarName.split("_")[1])
        v2 = int(variable.VarName.split("_")[2])
        cut_edges.add((v1,v2))
print(f"Cut edges: {cut_edges}")

The following code plots the graph from above colouring the nodes into the partition. Cut edges are highlighted.

In [ ]:
# Nodes
nx.draw_networkx_nodes(G, pos, nodelist=list(S_set), node_color="tab:red")
nx.draw_networkx_nodes(G, pos, nodelist=list(not_S_set), node_color="tab:blue")

# Edges
nx.draw_networkx_edges(G, pos, width=1.0, alpha=0.5)
nx.draw_networkx_edges(G, pos, edgelist=list(cut_edges), width=5, alpha=0.5, edge_color="tab:red")

# Plot figure
plt.tight_layout()
plt.axis("off")
plt.show()

# Semidefinite relaxation

The SDP relation of the maximum cut problem looks as follows:

$\max f(v) = \frac{1}{4}\sum_{i = 1}^n \sum_{j = 1}^n w_{ij} (1 - \overline{\underline{X}}_{ij})$,

where $w_{ij}$ is the weight of the edge connecting node $i$ with node $j$.

Subject to:

$\overline{\underline{X}} \in S_+^n,$

$\overline{\underline{X}}_{ii} = 1, \quad \forall i \in \{1, \dots, n\}.$

## Model building and solving (SDP to QCQP)

The following code builds and solves the approximation approach using the Gurobi API for Python. Note that since SDPs are not supported by Gurobi we work with the knowledge that $\overline{\underline{X}}^\ast = N \cdot N^T$ and work on solving entries of $N$.

In [10]:
# Initialise model
model = gp.Model()

# Adding matrix of variables
N = model.addMVar((number_of_nodes,number_of_nodes), name="n", vtype="c", lb = -float('inf'))

# Set up the quadratic expression for constraining
X = N.T @ N
for i in range(number_of_nodes):
    model.addConstr(X[i,i] == 1)
    for j in range(number_of_nodes):
        model.addConstr(X[i,j] == X[j,i])

# Adding the objective function
objective_sum = gp.LinExpr()
for i in range(number_of_nodes):
    for j in range(number_of_nodes):
        if i == j:
            continue
        if (i,j) in G.edges or (j,i) in G.edges:
            if i <= j:
                weight = G.get_edge_data(i,j)["weight"]
            else:
                weight = G.get_edge_data(j,i)["weight"]
            objective_sum += weight*(1-X[i,j])
model.setObjective(0.5*objective_sum, gp.GRB.MAXIMIZE)

# Optimize model
model.setParam('TimeLimit', 120)
model.optimize()

for i in range(number_of_nodes):
    print([round(float(N[i,j].X),2) for j in range(number_of_nodes)])

Set parameter TimeLimit to value 120
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  120

Optimize a model with 0 rows, 100 columns and 0 nonzeros (Max)
Model fingerprint: 0x6c0ad2c3
Model has 0 linear objective coefficients and an objective constant of -131
Model has 230 quadratic objective terms
Model has 110 quadratic constraints
Coefficient statistics:
  Matrix range     [0e+00, 0e+00]
  QMatrix range    [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  QObjective range [2e+00, 2e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [0e+00, 0e+00]
  QRHS range       [1e+00, 1e+00]


Continuous model is non-convex -- solving as a MIP


Presolve time: 0.00s
Presolved: 771 rows, 432 columns, 2211 nonzeros
Presolved model has 330 bilinear 